# Suite PYPRT — Printing Python

The versions print any Python syntax node: a module as a file, anything else as the text it stands for. Trees built by hand
print as source text without help: parentheses are added where Python's grammar needs them, which is also where an
assignment expression, a `yield` or a lambda must be parenthesized. Printing validates the tree and checks it against
the version first.

In [ ]:
import { Errors, Syntax as F } from "@mbse/programs/Framework";
import { Python312, Python314, Syntax as S } from "@mbse/programs/Python";
import { PythonStandard } from "@mbse/programs/Python/_Standard";

import { equal, raises } from "./support.js";

const { PrintError } = Errors;
const PY315 = new PythonStandard(3, 15);
const P = (node: F.SyntaxNode) => PY315.print(node);
const n = (spelling: string) => new S.Identifier({ spelling });
const v = (spelling: string) => new S.Name({ id: n(spelling) });
const c = (spelling: string) => new S.Constant({ spelling });
const b = (left: S.Expression, op: S.BinaryOperator, right: S.Expression) => new S.BinOp({ left, op, right });
const u = (op: S.UnaryOperator, operand: S.Expression) => new S.UnaryOp({ op, operand });
const pas = () => [new S.Pass()];
const expr = (value: S.Expression) => new S.Expr({ value });
const compare = () => new S.Compare({ left: v("a"), comparisons: [new S.Comparison({ op: "<", comparator: v("b") })] });
const both = () => new S.BoolOp({ op: "and", values: [v("a"), v("b")] });
const lam = () => new S.Lambda({ args: new S.Arguments(), body: v("a") });
const ifexp = () => new S.IfExp({ body: v("a"), test: v("b"), orelse: v("c") });
const walrus = () => new S.NamedExpr({ target: v("x"), value: v("a") });
const yielded = () => new S.Yield({ value: v("a") });
const pair = () => new S.Tuple({ elts: [v("a"), v("b")] });
const fn = () => new S.FunctionDef({ name: n("f"), args: new S.Arguments(), body: pas() });
const alternatives = () => new S.MatchOr({ patterns: [new S.MatchValue({ value: c("1") }), new S.MatchValue({ value: c("2") })] });
const captured = () => new S.MatchAs({ pattern: new S.MatchAs({ name: n("a") }), name: n("b") });
const openSequence = () => new S.MatchSequence({ patterns: [new S.MatchAs({ name: n("a") })] });

## PYPRT-01 · Precedence: operands are parenthesized where they bind less than their place needs

In [ ]:
equal(P(b(b(v("a"), "+", v("b")), "*", v("c"))), "(a + b) * c");
equal(P(b(v("a"), "*", b(v("b"), "+", v("c")))), "a * (b + c)");
equal(P(b(v("a"), "-", b(v("b"), "-", v("c")))), "a - (b - c)");
equal(P(b(b(v("a"), "-", v("b")), "-", v("c"))), "a - b - c");
equal(P(b(v("a"), "**", b(v("b"), "**", v("c")))), "a ** b ** c");
equal(P(b(b(v("a"), "**", v("b")), "**", v("c"))), "(a ** b) ** c");
equal(P(b(u("-", v("a")), "**", v("b"))), "(-a) ** b");
equal(P(u("-", b(v("a"), "**", v("b")))), "-a ** b");
equal(P(b(v("a"), "**", u("-", v("b")))), "a ** -b");
equal(P(b(new S.Await({ value: v("a") }), "**", v("b"))), "await a ** b");
equal(P(new S.Await({ value: b(v("a"), "**", v("b")) })), "await (a ** b)");
equal(P(new S.Await({ value: new S.Await({ value: v("a") }) })), "await (await a)");
equal(P(b(b(v("a"), "^", v("b")), "&", v("c"))), "(a ^ b) & c");
equal(P(b(v("a"), "|", b(v("b"), "^", v("c")))), "a | b ^ c");
equal(P(u("not", u("not", v("a")))), "not not a");
equal(P(u("-", u("not", v("a")))), "-(not a)");
equal(P(new S.Compare({ left: compare(), comparisons: [new S.Comparison({ op: "is not", comparator: compare() })] })),
  "(a < b) is not (a < b)");
equal(P(u("not", compare())), "not a < b");
equal(P(new S.BoolOp({ op: "and", values: [both(), v("c")] })), "(a and b) and c");
equal(P(new S.BoolOp({ op: "or", values: [both(), u("not", v("c"))] })), "a and b or not c");
equal(P(new S.BoolOp({ op: "and", values: [new S.BoolOp({ op: "or", values: [v("a"), v("b")] }), v("c")] })),
  "(a or b) and c");
equal(P(new S.IfExp({ body: ifexp(), test: ifexp(), orelse: ifexp() })),
  "(a if b else c) if (a if b else c) else a if b else c");
equal(P(new S.IfExp({ body: lam(), test: v("b"), orelse: lam() })), "(lambda: a) if b else lambda: a");
equal(P(new S.Lambda({ args: new S.Arguments(), body: ifexp() })), "lambda: a if b else c");
equal(P(expr(walrus())), "(x := a)");
equal(P(new S.Assign({ targets: [v("y")], value: walrus() })), "y = (x := a)");
equal(P(new S.If({ test: walrus(), body: pas() })), "if x := a:\n    pass");
equal(P(new S.Call({ func: v("f"), args: [walrus()], keywords: [new S.Keyword({ arg: n("k"), value: walrus() })] })),
  "f(x := a, k=(x := a))");
equal(P(new S.NamedExpr({ target: v("x"), value: new S.NamedExpr({ target: v("y"), value: v("a") }) })), "x := (y := a)");
equal(P(new S.List({ elts: [walrus()] })), "[x := a]");
equal(P(new S.Tuple({ elts: [walrus(), v("b")] })), "(x := a), b");
equal(P(new S.Parenthesized({ value: new S.Tuple({ elts: [walrus(), v("b")] }) })), "(x := a, b)");
equal(P(expr(yielded())), "yield a");
equal(P(new S.Assign({ targets: [v("x")], value: yielded() })), "x = yield a");
equal(P(new S.Return({ value: yielded() })), "return (yield a)");
equal(P(new S.Call({ func: v("f"), args: [yielded()] })), "f((yield a))");
equal(P(new S.Parenthesized({ value: yielded() })), "(yield a)");
equal(P(new S.Return({ value: new S.Tuple({ elts: [v("a"), v("b")] }) })), "return a, b");
equal(P(new S.Call({ func: v("f"), args: [pair()] })), "f((a, b))");
equal(P(new S.Tuple({ elts: [pair()] })), "(a, b),");
equal(P(new S.Starred({ value: b(v("a"), "+", v("b")) })), "*a + b");
equal(P(new S.Starred({ value: both() })), "*(a and b)");
equal(P(new S.Attribute({ value: b(v("a"), "+", v("b")), attr: n("c") })), "(a + b).c");
equal(P(new S.Attribute({ value: c("1"), attr: n("real") })), "(1).real");
equal(P(new S.Attribute({ value: c("1.0"), attr: n("real") })), "1.0.real");
equal(P(new S.Subscript({ value: new S.Call({ func: v("f") }), slice: pair() })), "f()[a, b]");
equal(P(new S.Subscript({ value: v("a"), slice: new S.Tuple({ elts: [v("i")] }) })), "a[i,]");
equal(P(new S.Subscript({ value: v("a"), slice: new S.Tuple() })), "a[()]");
console.log("ok");

## PYPRT-02 · Every kind prints on its own, as the text it stands for

In [ ]:
const generator = (target: S.Expression, iter: S.Expression) => new S.Comprehension({ target, iter });
const args = new S.Arguments({ posonlyargs: [new S.Arg({ arg: n("a") })],
  args: [new S.Arg({ arg: n("b"), annotation: v("int"), default_value: c("1") })], vararg: new S.Arg({ arg: n("c") }),
  kwonlyargs: [new S.Arg({ arg: n("d"), default_value: c("2") })], kwarg: new S.Arg({ arg: n("e") }) });
const texts: [F.SyntaxNode, string][] = [
  [n("x"), "x"], [new S.DottedName({ names: [n("a"), n("b")] }), "a.b"], [c("0x_FF"), "0x_FF"],
  [new S.ConcatenatedString({ values: [c("'a'"), c("\"b\"")] }), "'a' \"b\""],
  [new S.StringText({ spelling: "{{a}}" }), "{{a}}"],
  [new S.FormatSpec({ values: [new S.StringText({ spelling: ">10" })] }), ":>10"],
  [new S.FormattedValue({ value: v("x") }), "{x}"], [new S.Interpolation({ value: v("x"), conversion: "r" }), "{x!r}"],
  [new S.JoinedStr({ prefix: "f", quote: "'", values: [new S.StringText({ spelling: "a" })] }), "f'a'"],
  [new S.TemplateStr({ prefix: "t", quote: "\"\"\"", values: [] }), "t\"\"\"\"\"\""],
  [new S.Parenthesized({ value: v("a") }), "(a)"], [new S.Tuple(), "()"], [new S.Tuple({ elts: [v("a")] }), "a,"],
  [new S.List(), "[]"], [new S.Set({ elts: [v("a")] }), "{a}"], [new S.Dict(), "{}"],
  [new S.DictItem({ key: v("k"), value: v("v") }), "k: v"], [new S.DictItem({ value: v("m") }), "**m"],
  [new S.Comprehension({ target: v("x"), iter: v("y"), ifs: [v("x"), u("not", v("x"))] }), "for x in y if x if not x"],
  [new S.Comprehension({ is_async: true, target: pair(), iter: pair() }), "async for a, b in (a, b)"],
  [new S.ListComp({ elt: v("x"), generators: [generator(v("x"), v("y"))] }), "[x for x in y]"],
  [new S.SetComp({ elt: v("x"), generators: [generator(v("x"), v("y"))] }), "{x for x in y}"],
  [new S.DictComp({ key: v("k"), value: v("v"), generators: [generator(v("k"), v("y"))] }), "{k: v for k in y}"],
  [new S.DictComp({ key: v("d"), generators: [generator(v("d"), v("y"))] }), "{**d for d in y}"],
  [new S.GeneratorExp({ elt: v("x"), generators: [generator(v("x"), ifexp())] }), "(x for x in (a if b else c))"],
  [new S.Yield(), "yield"], [new S.YieldFrom({ value: v("a") }), "yield from a"],
  [new S.Slice(), ":"], [new S.Slice({ lower: v("a"), step: v("c") }), "a::c"], [new S.Slice({ upper: v("b") }), ":b"],
  [new S.Keyword({ arg: n("k"), value: v("v") }), "k=v"], [new S.Keyword({ value: v("m") }), "**m"],
  [new S.Call({ func: v("f"), args: [new S.GeneratorExp({ elt: v("x"), generators: [generator(v("x"), v("y"))] })] }),
    "f(x for x in y)"],
  [new S.Call({ func: v("f"), args: [new S.GeneratorExp({ elt: v("x"), generators: [generator(v("x"), v("y"))] })],
    keywords: [new S.Keyword({ arg: n("k"), value: v("v") })] }), "f((x for x in y), k=v)"],
  [new S.Comparison({ op: "not in", comparator: v("a") }), "not in a"],
  [args, "a, /, b: int = 1, *c, d=2, **e"],
  [new S.Lambda({ args: new S.Arguments({ args: [new S.Arg({ arg: n("a"), default_value: c("1") })],
    vararg: new S.Arg({ arg: n("b") }) }), body: v("a") }), "lambda a=1, *b: a"],
  [new S.Arguments({ kwonlyargs: [new S.Arg({ arg: n("d") })] }), "*, d"], [new S.Arguments(), ""],
  [new S.Arg({ arg: n("a"), annotation: v("int") }), "a: int"], [new S.Alias(), "*"],
  [new S.Alias({ name: new S.DottedName({ names: [n("a"), n("b")] }), asname: n("c") }), "a.b as c"],
  [new S.WithItem({ context_expr: v("a"), optional_vars: pair() }), "a as (a, b)"],
  [new S.TypeVar({ name: n("T"), bound: v("int"), default_value: v("str") }), "T: int = str"],
  [new S.ParamSpec({ name: n("P"), default_value: v("a") }), "**P = a"], [new S.TypeVarTuple({ name: n("Ts") }), "*Ts"],
  [new S.MatchValue({ value: c("1") }), "1"], [new S.MatchSingleton({ value: "None" }), "None"], [new S.MatchStar(), "*_"],
  [new S.MatchAs(), "_"], [new S.MatchAs({ name: n("a") }), "a"], [new S.MatchSequence(), "()"],
  [new S.MatchMapping({ keys: [c("1")], patterns: [new S.MatchAs()], rest: n("r") }), "{1: _, **r}"],
  [new S.MatchClass({ cls: v("P"), patterns: [new S.MatchAs()], kwd_attrs: [n("x")],
    kwd_patterns: [new S.MatchStar({ name: n("s") })] }), "P(_, x=*s)"],
  [new S.Comment({ text: " a" }), "# a"], [new S.Pass(), "pass"], [new S.Break(), "break"], [new S.Continue(), "continue"],
  [new S.Global({ names: [n("a"), n("b")] }), "global a, b"], [new S.Nonlocal({ names: [n("a")] }), "nonlocal a"],
  [new S.Import({ is_lazy: true, names: [new S.Alias({ name: new S.DottedName({ names: [n("a")] }) })] }), "lazy import a"],
  [new S.ImportFrom({ is_lazy: true, level: 2n, names: [new S.Alias()] }), "lazy from .. import *"],
  [new S.Raise(), "raise"], [new S.Raise({ exc: v("E"), cause: v("c") }), "raise E from c"],
  [new S.Delete({ targets: [v("a"), new S.Subscript({ value: v("b"), slice: c("0") })] }), "del a, b[0]"],
  [new S.AnnAssign({ target: v("a"), annotation: lam() }), "a: lambda: a"],
  [new S.AugAssign({ target: v("a"), op: "<<", value: yielded() }), "a <<= yield a"],
  [new S.TypeAlias({ name: v("A"), value: v("int") }), "type A = int"],
  [new S.ExceptHandler({ body: pas() }), "except:\n    pass"],
  [new S.MatchCase({ pattern: new S.MatchAs(), guard: walrus(), body: pas() }), "case _ if x := a:\n    pass"],
  [new S.Module(), ""],
];
for (const [node, text] of texts) equal(P(node), text);
console.log(texts.length, "texts");

## PYPRT-03 · Statements: one per line, blocks indented, `elif`, blank lines around definitions, and comments

In [ ]:
const klass = new S.ClassDef({ decorator_list: [v("d")], name: n("C"), body: [expr(c("'doc'")), new S.FunctionDef({
  name: n("m"), args: new S.Arguments({ args: [new S.Arg({ arg: n("self") })] }), body: pas() }), new S.Pass()] });
const module = new S.Module({ body: [new S.Import({ names: [new S.Alias({ name: new S.DottedName({ names: [n("os")] }) })] }),
  new S.Comment({ text: " lead" }), fn(), klass, new S.Comment({ text: " trailing", trailing: true }),
  new S.Assign({ targets: [v("x")], value: c("1") })] });
equal(P(module), "import os\n\n\n# lead\ndef f():\n    pass\n\n\n@d\nclass C:\n    'doc'\n\n"
  + "    def m(self):\n        pass\n\n    pass  # trailing\n\n\nx = 1\n");
const chain = new S.If({ test: v("a"), body: pas(), orelse: [new S.If({ test: v("b"),
  body: [new S.Comment({ text: " c", trailing: true }), new S.Pass()], orelse: [new S.If({ test: v("c"), body: pas() })] })] });
equal(P(chain), "if a:\n    pass\nelif b:  # c\n    pass\nelif c:\n    pass");
const nested = new S.If({ test: v("a"), body: pas(),
  orelse: [new S.Comment({ text: " keeps else" }), new S.If({ test: v("b"), body: pas() })] });
equal(P(nested), "if a:\n    pass\nelse:\n    # keeps else\n    if b:\n        pass");
const loops = [new S.While({ test: v("a"), body: pas(), orelse: pas() }), new S.For({ target: v("x"), iter: pair(), body: pas() }),
  new S.AsyncFor({ target: v("x"), iter: v("y"), body: pas(), orelse: pas() }), new S.While({ test: v("a"), body: pas() })];
equal(loops.map(P), ["while a:\n    pass\nelse:\n    pass", "for x in a, b:\n    pass",
  "async for x in y:\n    pass\nelse:\n    pass", "while a:\n    pass"]);
const tried = new S.TryStar({ body: pas(), handlers: [new S.ExceptHandler({ type: pair(), body: pas() }),
  new S.ExceptHandler({ type: pair(), name: n("e"), body: pas() })], orelse: pas(), finalbody: pas() });
equal(P(tried), "try:\n    pass\nexcept* a, b:\n    pass\nexcept* (a, b) as e:\n    pass\nelse:\n    pass\n"
  + "finally:\n    pass");
equal(P(new S.With({ items: [new S.WithItem({ context_expr: v("a") })], body: pas() })), "with a:\n    pass");
equal(P(new S.AsyncWith({ items: [new S.WithItem({ context_expr: lam(), optional_vars: v("b") })], body: pas() })),
  "async with lambda: a as b:\n    pass");
const generic = new S.AsyncFunctionDef({ name: n("f"), type_params: [new S.TypeVar({ name: n("T") })],
  args: new S.Arguments(), returns: v("T"), body: pas() });
equal(P(generic), "async def f[T]() -> T:\n    pass");
equal(P(new S.ClassDef({ name: n("C"), bases: [v("B")], keywords: [new S.Keyword({ arg: n("m"), value: v("M") })],
  body: pas() })), "class C(B, m=M):\n    pass");
equal(P(new S.Match({ subject: pair(), cases: [new S.MatchCase({
  pattern: new S.MatchSequence({ patterns: [new S.MatchAs(), new S.MatchAs()] }), body: pas() })] })),
"match a, b:\n    case _, _:\n        pass");
equal(P(new S.Module({ body: [new S.Comment({ text: " only", trailing: true })] })), "# only\n");
equal(P(new S.Module({ body: [new S.Comment({ text: " first" }), fn()] })), "# first\ndef f():\n    pass\n");
equal(P(new S.Module({ body: [new S.Assign({ targets: [v("x")], value: c("1") }), new S.Comment({ text: " t", trailing: true }),
  new S.Comment({ text: " lead" }), fn()] })), "x = 1  # t\n\n\n# lead\ndef f():\n    pass\n");
console.log("ok");

## PYPRT-04 · f-strings: a replacement field prints its text while that spells its value, and its value otherwise

In [ ]:
const joined = (...values: (S.StringText | S.FormattedValue)[]) => new S.JoinedStr({ prefix: "f", quote: "'", values });

const field = new S.FormattedValue({ value: b(v("a"), "+", v("b")), text: "a+b" });
equal(P(joined(field)), "f'{a+b}'");
field.value = v("c"); // the text no longer spells it
equal(P(joined(field)), "f'{c}'");
const debug = new S.FormattedValue({ value: v("x"), text: " x = ", debug: true, conversion: "r",
  format_spec: new S.FormatSpec({ values: [new S.StringText({ spelling: ">" }), new S.FormattedValue({ value: v("w") })] }) });
equal(P(joined(debug)), "f'{ x = !r:>{w}}'");
debug.text = "x = # a comment\n";
equal(P(joined(debug)), "f'{x = # a comment\n!r:>{w}}'");
debug.text = " y = ";
debug.value = v("x");
equal(P(joined(debug)), "f'{x=!r:>{w}}'");
equal(P(joined(new S.FormattedValue({ value: lam() }))), "f'{(lambda: a)}'");
equal(P(joined(new S.FormattedValue({ value: walrus() }))), "f'{(x := a)}'");
equal(P(joined(new S.FormattedValue({ value: new S.Set({ elts: [v("a")] }) }))), "f'{ {a}}'");
equal(P(joined(new S.FormattedValue({ value: pair() }))), "f'{a, b}'");
equal(P(joined(new S.FormattedValue({ value: yielded() }))), "f'{yield a}'");
equal(P(joined(new S.FormattedValue({ value: v("x"), text: "(x)" }))), "f'{x}'");
const commented = new S.FormattedValue({ value: c("\"\"\"a # b\"\"\""), text: " # a comment\n  \"\"\"a # b\"\"\"=", debug: true });
equal(P(new S.JoinedStr({ prefix: "f", quote: "\"", values: [commented] })), "f\"{ # a comment\n  \"\"\"a # b\"\"\"=}\"");
const spaced = new S.FormattedValue({ value: c("'a b'"), text: "'ab'" }); // whitespace in a string is code
equal(P(new S.JoinedStr({ prefix: "f", quote: "\"", values: [spaced] })), "f\"{'a b'}\"");
const escaped = new S.FormattedValue({ value: c("'a\\'b'"), text: " 'a\\'b' " });
equal(P(new S.JoinedStr({ prefix: "f", quote: "\"", values: [escaped] })), "f\"{ 'a\\'b' }\"");
console.log("ok");

## PYPRT-05 · Patterns: parentheses where a pattern needs them, and open sequences only in a case

In [ ]:
const kase = (pattern: S.Pattern) =>
  (P(new S.Match({ subject: v("x"), cases: [new S.MatchCase({ pattern, body: pas() })] })).split("\n")[1] as string).slice(9, -1);

equal(kase(new S.MatchAs({ pattern: alternatives(), name: n("s") })), "1 | 2 as s");
equal(kase(new S.MatchAs({ pattern: captured(), name: n("c") })), "(a as b) as c");
equal(kase(new S.MatchAs({ pattern: alternatives() })), "1 | 2 as _");
equal(kase(new S.MatchOr({ patterns: [alternatives(), captured()] })), "(1 | 2) | (a as b)");
equal(kase(openSequence()), "a,");
equal(kase(new S.MatchSequence({ patterns: [openSequence(), openSequence()] })), "(a,), (a,)");
equal(kase(new S.MatchSequence({ delimiters: "()", patterns: [new S.MatchAs({ name: n("a") })] })), "(a,)");
equal(kase(new S.MatchSequence({ delimiters: "[]", patterns: [openSequence()] })), "[(a,)]");
equal(kase(new S.MatchMapping({ keys: [u("-", c("1"))], patterns: [openSequence()] })), "{-1: (a,)}");
equal(kase(new S.MatchSequence({ delimiters: "()" })), "()");
equal(kase(new S.MatchSequence({ patterns: [new S.MatchSequence()] })), "(),");
console.log("ok");

## PYPRT-06 · Printing validates the tree and checks it against the version

In [ ]:
raises(PrintError, () => P(new S.If({ body: pas() })), "an If needs a test");
raises(PrintError, () => P(v("class")), "'class' is a keyword");
raises(PrintError, () => Python312.print(new S.TemplateStr({ prefix: "t", quote: "'" })),
  "TemplateStr needs Python 3.14, but this is Python 3.12");
raises(PrintError, () => Python314.print(new S.Module({ body: [new S.Import({ is_lazy: true,
  names: [new S.Alias({ name: new S.DottedName({ names: [n("a")] }) })] })] })),
"body[0]: Import.is_lazy needs Python 3.15, but this is Python 3.14");
equal(Python312.print(new S.Module({ body: [expr(walrus())] })), "(x := a)\n");
equal(P(new S.Return({ value: walrus() })), "return (x := a)");
equal(P(new S.For({ target: v("x"), iter: walrus(), body: pas() })), "for x in (x := a):\n    pass");
console.log("ok");